<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.7_Praktikum_Video_Processing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.7: Praktikum Video Processing
**Mata Kuliah:** Visi Komputer dan Pengolahan Citra Digital  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Membangkitkan berkas video simulasi dan mengonfigurasi `cv2.VideoWriter` dengan FourCC codec `mp4v`.
2. Membaca aliran video frame per frame menggunakan `cv2.VideoCapture`.
3. Melakukan pengukuran dan pencatatan laju bingkai aktual (*FPS Profiling*) menggunakan `time.perf_counter()`.
4. Menyematkan teks anotasi *On-Screen Display* (OSD) dan menyimpan video hasil olahan.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')


## 1. Pembangkitan Video Sintetis Aliran Konveyor TBS Sawit
Membangkitkan video berdurasi 60 frame (2 detik pada 30 FPS) yang mensimulasikan buah sawit melintas di sabuk berjalan.

In [ ]:
video_path = 'conveyor_synthetic.mp4'
fps = 30.0
W, H = 640, 480
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = cv2.VideoWriter(video_path, fourcc, fps, (W, H))

for i in range(60):
    frame = np.full((H, W, 3), 45, dtype=np.uint8)
    # Buah melintas dari kiri ke kanan
    bx = int(60 + i * 8.5)
    by = 240
    cv2.circle(frame, (bx, by), 40, (20, 110, 220), -1)
    cv2.putText(frame, f'Conveyor Frame: #{i+1}', (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    out_writer.write(frame)

out_writer.release()
print(f'[OK] Berkas video berhasil disimpan: {video_path}')


## 2. Pemrosesan Video dan Profiling FPS Nyata
Membaca frame video, menerapkan deteksi sederhana, mengukur FPS aktual, dan menyimpan video teranotasi.

In [ ]:
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), 'Gagal membuka video!'

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Total Frame Video: {total_frames}')

processed_path = 'conveyor_processed.mp4'
processed_writer = cv2.VideoWriter(processed_path, fourcc, fps, (W, H))

fps_history = []
t_prev = time.perf_counter()
sample_frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    # Simulasi komputasi inspeksi buah
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 70, 255, cv2.THRESH_BINARY)
    
    # Profiling FPS
    t_curr = time.perf_counter()
    dt = t_curr - t_prev
    t_prev = t_curr
    inst_fps = 1.0 / dt if dt > 0 else 0.0
    fps_history.append(inst_fps)
    
    # Anotasi OSD
    cv2.putText(frame, f'FPS: {inst_fps:.1f}', (W - 140, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    processed_writer.write(frame)
    
    if len(sample_frames) < 3 and len(fps_history) in [15, 30, 45]:
        sample_frames.append(frame.copy())

cap.release()
processed_writer.release()

avg_fps = np.mean(fps_history[1:])
print(f'Selesai memproses {len(fps_history)} frame! Rata-rata FPS: {avg_fps:.1f}')


## 3. Visualisasi Hasil Pengukuran FPS dan Sampel Frame Video
Menampilkan grafik stabilitas laju bingkai dan cuplikan frame video hasil olahan.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(fps_history[1:], color='dodgerblue', linewidth=2)
ax1.axhline(30.0, color='crimson', linestyle='--', label='Target 30 FPS')
ax1.set_title('Dinamika Profiling FPS Video Processing', fontsize=11, fontweight='bold')
ax1.set_xlabel('Nomor Frame')
ax1.set_ylabel('Laju Bingkai (FPS)')
ax1.legend()

if sample_frames:
    ax2.imshow(cv2.cvtColor(sample_frames[-1], cv2.COLOR_BGR2RGB))
    ax2.set_title('Cuplikan Frame Video Teranotasi OSD', fontsize=11, fontweight='bold')
    ax2.axis('off')

plt.tight_layout()
plt.savefig('praktikum_video_processing_11_7.png', dpi=150)
plt.show()
